# ADFA-LD 

We propose to first test the GNN methods over the AFDA-LD dataset 

In [7]:
import os
from pathlib import Path
from collections import Counter

import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

We first define the dataLoader class of the dataset, it permits to to read the dataset file correctly, 

class ADFALDLoader:
    def __init__(self, root_dir):
        self.root = Path(root_dir)

    def read_trace_file(self, filepath):
        """
        Read one syscall trace file.
        Returns list[int]
        """
        with open(filepath, "r") as f:
            content = f.read().strip() # sert à quoi le .strip() ? 

        # syscall IDs separated by spaces
        seq = list(map(int, content.split())) # dcp c'est quoi exactement? le map fait quoi? 
        return seq

    def load_normal_training(self):
        folder = self.root / "Training_Data_Master" 
        data = []

        for file in folder.glob("*"):
            seq = self.read_trace_file(file)
            data.append({
                "file": file.name,
                "label": 0,
                "attack_type": None,
                "sequence": seq
            }) # donc chaque file corespond à quoi? une séquence syscall, qui peut être sous attack ou non? 

        return pd.DataFrame(data)

    def load_normal_validation(self):
        folder = self.root / "Validation_Data_Master"
        data = []

        for file in folder.glob("*"):
            seq = self.read_trace_file(file)
            data.append({
                "file": file.name,
                "label": 0,
                "attack_type": None,
                "sequence": seq
            })

        return pd.DataFrame(data) # ? pourquoi on fait une normal validation et un training en label 0, et toutes les attack d'un autre? 

    def load_attacks(self):
        folder = self.root / "Attack_Data_Master"
        data = []

        for attack_dir in folder.iterdir():
            if attack_dir.is_dir():
                attack_name = attack_dir.name

                for file in attack_dir.glob("*"):
                    seq = self.read_trace_file(file)

                    data.append({
                        "file": file.name,
                        "label": 1,
                        "attack_type": attack_name,
                        "sequence": seq
                    })

        return pd.DataFrame(data)

    def load_all(self):
        df_train = self.load_normal_training()
        df_val = self.load_normal_validation()
        df_attack = self.load_attacks()

        return pd.concat([df_train, df_val, df_attack], ignore_index=True)

In [8]:
class ADFALDLoader:
    """
    Expected ADFA-LD structure:

    root/
        Training_Data_Master/
            *.txt or files without extension
        Validation_Data_Master/
            *.txt or files without extension
        Attack_Data_Master/
            AttackType1/
                *.txt
            AttackType2/
                *.txt
            ...
    """

    def __init__(self, root_dir):
        self.root = Path(root_dir)

        self.train_normal_dir = self.root / "Training_Data_Master"
        self.val_normal_dir = self.root / "Validation_Data_Master"
        self.attack_dir = self.root / "Attack_Data_Master"

    @staticmethod
    def read_trace_file(file_path):
        """
        Robust reader for ADFA-LD traces.
        Works whether integers are space-separated, newline-separated, or mixed.
        """
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read().strip()

        if not content:
            return []

        tokens = content.split()
        seq = [int(tok) for tok in tokens]
        return seq

    def load_normal_training(self):
        rows = []
        if not self.train_normal_dir.exists():
            raise FileNotFoundError(f"Missing directory: {self.train_normal_dir}")

        for file_path in sorted(self.train_normal_dir.glob("*")):
            if file_path.is_file():
                seq = self.read_trace_file(file_path)
                if len(seq) > 1:
                    rows.append({
                        "file": str(file_path),
                        "sequence": seq,
                        "label": 0,
                        "attack_type": "normal_train",
                        "split_hint": "train"
                    })
        return rows

    def load_normal_validation(self):
        rows = []
        if not self.val_normal_dir.exists():
            raise FileNotFoundError(f"Missing directory: {self.val_normal_dir}")

        for file_path in sorted(self.val_normal_dir.glob("*")):
            if file_path.is_file():
                seq = self.read_trace_file(file_path)
                if len(seq) > 1:
                    rows.append({
                        "file": str(file_path),
                        "sequence": seq,
                        "label": 0,
                        "attack_type": "normal_val",
                        "split_hint": "val"
                    })
        return rows

    def load_attacks(self):
        rows = []
        if not self.attack_dir.exists():
            raise FileNotFoundError(f"Missing directory: {self.attack_dir}")

        for attack_subdir in sorted(self.attack_dir.iterdir()):
            if attack_subdir.is_dir():
                attack_name = attack_subdir.name
                for file_path in sorted(attack_subdir.glob("*")):
                    if file_path.is_file():
                        seq = self.read_trace_file(file_path)
                        if len(seq) > 1:
                            rows.append({
                                "file": str(file_path),
                                "sequence": seq,
                                "label": 1,
                                "attack_type": attack_name,
                                "split_hint": "attack"
                            })
        return rows

    def load_all(self):
        rows = []
        rows.extend(self.load_normal_training())
        rows.extend(self.load_normal_validation())
        rows.extend(self.load_attacks())
        return rows

Let's use that to load the dataset ! 

In [9]:
import pandas as pd

loader = ADFALDLoader("/Users/tristan/Documents/dev/01_Cours_CS/08_MLNS/intrusion-detection/data/ADFA-LD")

rows = loader.load_all()

df = pd.DataFrame(rows)

print(df.head())
print(df["label"].value_counts())
print(df["attack_type"].value_counts())

                                                file  \
0  /Users/tristan/Documents/dev/01_Cours_CS/08_ML...   
1  /Users/tristan/Documents/dev/01_Cours_CS/08_ML...   
2  /Users/tristan/Documents/dev/01_Cours_CS/08_ML...   
3  /Users/tristan/Documents/dev/01_Cours_CS/08_ML...   
4  /Users/tristan/Documents/dev/01_Cours_CS/08_ML...   

                                            sequence  label   attack_type  \
0  [6, 6, 63, 6, 42, 120, 6, 195, 120, 6, 6, 114,...      0  normal_train   
1  [54, 175, 120, 175, 175, 3, 175, 175, 120, 175...      0  normal_train   
2  [6, 11, 45, 33, 192, 33, 5, 197, 192, 6, 33, 5...      0  normal_train   
3  [7, 174, 174, 5, 197, 197, 6, 13, 195, 4, 4, 1...      0  normal_train   
4  [11, 45, 33, 192, 33, 5, 197, 192, 6, 33, 5, 3...      0  normal_train   

  split_hint  
0      train  
1      train  
2      train  
3      train  
4      train  
label
0    5205
1     746
Name: count, dtype: int64
attack_type
normal_val       4372
normal_train      833
Hy

Then we need to convert our sequence to a transition graph 

In [10]:
def sequence_to_graph(sequence):
    """
    Build a directed weighted graph from a syscall sequence.

    Node = syscall ID
    Edge u->v = syscall v occurs right after syscall u
    Edge weight = transition count
    """
    G = nx.DiGraph()

    # Count syscall occurrences for node-level frequencies
    counts = Counter(sequence)
    total_calls = len(sequence)

    for syscall, c in counts.items():
        G.add_node(
            syscall,
            count=c,
            freq=c / total_calls
        )

    for i in range(len(sequence) - 1):
        u = sequence[i]
        v = sequence[i + 1]
        if G.has_edge(u, v):
            G[u][v]["weight"] += 1.0
        else:
            G.add_edge(u, v, weight=1.0)

    return G


We now have a graph representing system call, we then need to add features to each node

In [11]:
def add_structural_node_features(G):
    """
    Adds useful structural features to each node:
    - frequency in trace
    - in_degree / out_degree (unweighted)
    - weighted_in_degree / weighted_out_degree
    - pagerank
    - self_loop flag
    """
    if G.number_of_nodes() == 0:
        return G

    pagerank = nx.pagerank(G, weight="weight") if G.number_of_edges() > 0 else {n: 0.0 for n in G.nodes()}

    for n in G.nodes():
        in_deg = G.in_degree(n)
        out_deg = G.out_degree(n)
        in_wdeg = G.in_degree(n, weight="weight")
        out_wdeg = G.out_degree(n, weight="weight")
        self_loop = 1.0 if G.has_edge(n, n) else 0.0

        G.nodes[n]["in_deg"] = float(in_deg)
        G.nodes[n]["out_deg"] = float(out_deg)
        G.nodes[n]["in_wdeg"] = float(in_wdeg)
        G.nodes[n]["out_wdeg"] = float(out_wdeg)
        G.nodes[n]["pagerank"] = float(pagerank[n])
        G.nodes[n]["self_loop"] = float(self_loop)

    return G

Then build global syscall vocab 

In [12]:
def build_syscall_vocab(rows):
    """
    Builds a global mapping syscall_id -> index
    """
    unique_syscalls = sorted({s for row in rows for s in row["sequence"]})
    return {syscall: idx for idx, syscall in enumerate(unique_syscalls)}

Convert to pytorch geometric 

In [13]:
def graph_to_pyg_data(G, label, syscall_vocab, file_path=None, attack_type=None):
    """
    Convert a NetworkX DiGraph into a PyTorch Geometric Data object.

    Data fields:
    - x: numeric structural features
    - node_ids: syscall indices for learned embedding
    - edge_index
    - edge_weight
    - y
    """
    nodes = list(G.nodes())
    node_to_idx = {node: i for i, node in enumerate(nodes)}

    x_list = []
    node_ids = []

    for node in nodes:
        attrs = G.nodes[node]

        x_list.append([
            float(attrs.get("freq", 0.0)),
            float(attrs.get("in_deg", 0.0)),
            float(attrs.get("out_deg", 0.0)),
            float(attrs.get("in_wdeg", 0.0)),
            float(attrs.get("out_wdeg", 0.0)),
            float(attrs.get("pagerank", 0.0)),
            float(attrs.get("self_loop", 0.0)),
        ])

        node_ids.append(syscall_vocab[node])

    x = torch.tensor(x_list, dtype=torch.float)
    node_ids = torch.tensor(node_ids, dtype=torch.long)

    edges = []
    edge_weights = []

    for u, v, attrs in G.edges(data=True):
        edges.append([node_to_idx[u], node_to_idx[v]])
        edge_weights.append(float(attrs.get("weight", 1.0)))

    if len(edges) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_weight = torch.empty((0,), dtype=torch.float)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_weight = torch.tensor(edge_weights, dtype=torch.float)

    y = torch.tensor([label], dtype=torch.long)

    data = Data(
        x=x,
        node_ids=node_ids,
        edge_index=edge_index,
        edge_weight=edge_weight,
        y=y
    )

    data.file_path = file_path if file_path is not None else ""
    data.attack_type = attack_type if attack_type is not None else ""

    return data


def rows_to_pyg_dataset(rows, syscall_vocab):
    dataset = []
    for row in rows:
        G = sequence_to_graph(row["sequence"])
        G = add_structural_node_features(G)
        data = graph_to_pyg_data(
            G,
            label=row["label"],
            syscall_vocab=syscall_vocab,
            file_path=row["file"],
            attack_type=row["attack_type"]
        )
        dataset.append(data)
    return dataset

split train/val/test ...

In [14]:
def split_dataset(dataset, test_size=0.2, val_size=0.2, random_state=42):
    labels = [int(data.y.item()) for data in dataset]
    idx = np.arange(len(dataset))

    idx_trainval, idx_test = train_test_split(
        idx,
        test_size=test_size,
        random_state=random_state,
        stratify=labels
    )

    labels_trainval = [labels[i] for i in idx_trainval]

    relative_val_size = val_size / (1.0 - test_size)

    idx_train, idx_val = train_test_split(
        idx_trainval,
        test_size=relative_val_size,
        random_state=random_state,
        stratify=labels_trainval
    )

    train_dataset = [dataset[i] for i in idx_train]
    val_dataset = [dataset[i] for i in idx_val]
    test_dataset = [dataset[i] for i in idx_test]

    return train_dataset, val_dataset, test_dataset

## Model ! 

In [15]:
class SyscallGNN(nn.Module):
    def __init__(
        self,
        num_syscalls,
        num_numeric_features,
        syscall_emb_dim=32,
        hidden_dim=64,
        num_classes=2,
        dropout=0.3
    ):
        super().__init__()

        self.syscall_embedding = nn.Embedding(num_syscalls, syscall_emb_dim)

        input_dim = syscall_emb_dim + num_numeric_features

        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)

        self.dropout = dropout

        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, data):
        # data.node_ids: [num_nodes]
        # data.x: [num_nodes, num_numeric_features]
        emb = self.syscall_embedding(data.node_ids)
        x = torch.cat([emb, data.x], dim=1)

        x = self.conv1(x, data.edge_index, data.edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, data.edge_index, data.edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, data.edge_index, data.edge_weight)
        x = F.relu(x)

        # Graph-level pooling
        x = global_mean_pool(x, data.batch)

        out = self.mlp(x)
        return out

and train.. 

In [16]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for batch in loader:
        batch = batch.to(device)

        optimizer.zero_grad()
        logits = model(batch)
        loss = F.cross_entropy(logits, batch.y.view(-1))
        loss.backward()
        optimizer.step()

        batch_size = batch.num_graphs
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    return total_loss / total_examples

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    all_y_true = []
    all_y_pred = []
    all_y_prob = []

    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        probs = F.softmax(logits, dim=1)[:, 1]
        preds = logits.argmax(dim=1)

        all_y_true.extend(batch.y.view(-1).cpu().numpy().tolist())
        all_y_pred.extend(preds.cpu().numpy().tolist())
        all_y_prob.extend(probs.cpu().numpy().tolist())

    acc = accuracy_score(all_y_true, all_y_pred)
    f1 = f1_score(all_y_true, all_y_pred)

    try:
        auc = roc_auc_score(all_y_true, all_y_prob)
    except ValueError:
        auc = float("nan")

    return {
        "acc": acc,
        "f1": f1,
        "auc": auc,
        "y_true": all_y_true,
        "y_pred": all_y_pred,
        "y_prob": all_y_prob
    }

def fit_model(model, train_loader, val_loader, device, lr=1e-3, weight_decay=1e-4, epochs=30):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        train_metrics = evaluate(model, train_loader, device)
        val_metrics = evaluate(model, val_loader, device)

        print(
            f"Epoch {epoch:03d} | "
            f"loss={train_loss:.4f} | "
            f"train_acc={train_metrics['acc']:.4f} | "
            f"train_f1={train_metrics['f1']:.4f} | "
            f"val_acc={val_metrics['acc']:.4f} | "
            f"val_f1={val_metrics['f1']:.4f} | "
            f"val_auc={val_metrics['auc']:.4f}"
        )

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return model



In [19]:
def main():
    root_dir = "/Users/tristan/Documents/dev/01_Cours_CS/08_MLNS/intrusion-detection/data/ADFA-LD"   # <-- CHANGE THIS

    # Load traces
    loader = ADFALDLoader(root_dir)
    rows = loader.load_all()

    print(f"Number of traces: {len(rows)}")
    print(f"Normal: {sum(r['label'] == 0 for r in rows)}")
    print(f"Attack: {sum(r['label'] == 1 for r in rows)}")

    # Build global syscall vocabulary
    syscall_vocab = build_syscall_vocab(rows)
    print(f"Number of unique syscalls: {len(syscall_vocab)}")

    # Convert all traces to graphs / PyG Data objects
    dataset = rows_to_pyg_dataset(rows, syscall_vocab)

    # Split
    train_dataset, val_dataset, test_dataset = split_dataset(
        dataset,
        test_size=0.2,
        val_size=0.2,
        random_state=42
    )

    print(f"Train graphs: {len(train_dataset)}")
    print(f"Val graphs:   {len(val_dataset)}")
    print(f"Test graphs:  {len(test_dataset)}")

    # Dataloaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Model
    num_numeric_features = train_dataset[0].x.shape[1]
    model = SyscallGNN(
        num_syscalls=len(syscall_vocab),
        num_numeric_features=num_numeric_features,
        syscall_emb_dim=32,
        hidden_dim=64,
        num_classes=2,
        dropout=0.3
    ).to(device)

    # Train
    model = fit_model(
        model,
        train_loader,
        val_loader,
        device,
        lr=1e-3,
        weight_decay=1e-4,
        epochs=30
    )

    # Final evaluation
    test_metrics = evaluate(model, test_loader, device)
    print("\n=== TEST METRICS ===")
    print(f"Accuracy: {test_metrics['acc']:.4f}")
    print(f"F1-score: {test_metrics['f1']:.4f}")
    print(f"ROC-AUC:  {test_metrics['auc']:.4f}")

    print("\n=== CLASSIFICATION REPORT ===")
    print(classification_report(test_metrics["y_true"], test_metrics["y_pred"], digits=4))



In [20]:
main()

Number of traces: 5951
Normal: 5205
Attack: 746
Number of unique syscalls: 175
Train graphs: 3570
Val graphs:   1190
Test graphs:  1191
Using device: cpu
Epoch 001 | loss=0.4953 | train_acc=0.8571 | train_f1=0.3943 | val_acc=0.8622 | val_f1=0.4143 | val_auc=0.7522
Epoch 002 | loss=0.3509 | train_acc=0.8647 | train_f1=0.1390 | val_acc=0.8630 | val_f1=0.1189 | val_auc=0.8211
Epoch 003 | loss=0.3159 | train_acc=0.8633 | train_f1=0.2327 | val_acc=0.8672 | val_f1=0.2330 | val_auc=0.8362
Epoch 004 | loss=0.3056 | train_acc=0.8653 | train_f1=0.1721 | val_acc=0.8672 | val_f1=0.1856 | val_auc=0.8519
Epoch 005 | loss=0.3016 | train_acc=0.8641 | train_f1=0.2640 | val_acc=0.8664 | val_f1=0.2673 | val_auc=0.8597
Epoch 006 | loss=0.2760 | train_acc=0.8779 | train_f1=0.0644 | val_acc=0.8748 | val_f1=0.0387 | val_auc=0.8903
Epoch 007 | loss=0.2578 | train_acc=0.8846 | train_f1=0.4278 | val_acc=0.8798 | val_f1=0.4066 | val_auc=0.9010
Epoch 008 | loss=0.2453 | train_acc=0.8790 | train_f1=0.4667 | val_ac